## Setup

In [1]:
from google.colab import userdata
access_token = userdata.get('CASM-NER')

In [2]:
%%capture
!pip install transformers
!pip install sentencepiece
!pip install seqeval
!pip install datasets
# !pip install git+https://github.com/ay94/multilingual-ner.git

In [3]:
# from ner import evaluation

In [4]:
## Mount GDrive
from google.colab import drive
drive.mount('/content/drive/', force_remount=True)

## Imports
import os
import sys
import nltk
import time
import torch
import random
import subprocess
import numpy as np
import pandas as pd
import datetime as dt
from itertools import groupby
from tqdm.notebook import tqdm
from datasets import load_dataset
from transformers import pipeline
from collections import Counter, defaultdict
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForTokenClassification, AutoTokenizer
from seqeval.metrics import f1_score as seq_f1, precision_score as seq_precision, recall_score as seq_recall, classification_report as seq_classification
from sklearn.metrics import f1_score as skl_f1, precision_score as skl_precision, recall_score as skl_recall, classification_report as skl_classification

Mounted at /content/drive/


In [5]:
# Append the library files into the notebook system path for import
sys.path.append('/content/drive/Shareddrives/Machine Translation/Model benchmarking/Libraries/1.0.2')
# import custom library files
import ner, utils

## Load datasets

### wikiann

In [6]:
wikiann_label_map = {
    "O": 0,
    "B-PER": 1,
    "I-PER": 2,
    "B-ORG": 3,
    "I-ORG": 4,
    "B-LOC": 5,
    "I-LOC": 6
}

wikiann_sr = ner.ReadNERData()
wikiann_hr = ner.ReadNERData()

wikiann_sr_words, wikiann_sr_labels = wikiann_sr.read_dataset('wikiann', wikiann_label_map, lang='sr')
wikiann_hr_words, wikiann_hr_labels = wikiann_hr.read_dataset('wikiann', wikiann_label_map, lang='hr')

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/20000 [00:00<?, ? examples/s]

Generating test Split


...

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/20000 [00:00<?, ? examples/s]

Generating test Split


  0%|          | 0/10000 [00:00<?, ?it/s]

# Evaluate model

In [14]:
alignment = {
'I-org': 'I-ORG',
'B-misc': 'O',
'B-per': 'B-PER',
'B-deriv-per': 'O',
'B-org': 'B-ORG',
'B-loc': 'B-LOC',
'I-deriv-per': 'O',
'I-misc': 'O',
'I-loc': 'I-LOC',
'I-per': 'I-PER',
'O': 'O',
 }

model_name = "Andrija/M-bert-NER"
model_name_output = 'Andrija/M-bert-NER'
model_evaluation = ner.ModelEvaluation(
    model_name,
    alignment
)

In [15]:
model_evaluation.model.config.id2label

{0: 'I-org',
 1: 'B-misc',
 2: 'B-per',
 3: 'B-deriv-per',
 4: 'B-org',
 5: 'B-loc',
 6: 'I-deriv-per',
 7: 'I-misc',
 8: 'I-loc',
 9: 'I-per',
 10: 'O'}

### wikiann - serbian

In [16]:
data_name = "wikiann_sr"
wikiann_evaluation_output = model_evaluation.evaluate_model(wikiann_sr_words, wikiann_sr_labels)

  0%|          | 0/625 [00:00<?, ?it/s]

In [17]:
wikiann_seqeval = wikiann_evaluation_output.get_classification('Seqeval')
wikiann_seqeval

,Tag,Precision,Recall,F1,support
0,LOC,0.1472,0.1922,0.1667,3782
1,ORG,0.4788,0.3514,0.4053,3657
2,PER,0.5521,0.7528,0.6370,4139
3,micro,0.3865,0.4429,0.4128,11578
4,macro,0.3927,0.4321,0.4030,11578
5,weighted,0.3967,0.4429,0.4102,11578


In [18]:
wikiann_sklearn = wikiann_evaluation_output.get_classification('Sklearn')
wikiann_sklearn

,Tag,Precision,Recall,F1,support
0,B-LOC,0.3920,0.5037,0.4409,3782
1,B-ORG,0.5855,0.4195,0.4888,3657
2,B-PER,0.6401,0.8488,0.7298,4139
3,I-LOC,0.6229,0.0836,0.1474,9820
4,I-ORG,0.8847,0.2891,0.4358,7780
5,I-PER,0.8545,0.7221,0.7827,5962
6,O,0.7294,0.9908,0.8402,37048
7,accuracy,0.7069,72188,None,None
8,macro,0.6727,0.5511,0.5522,72188
9,weighted,0.7119,0.7069,0.6526,72188


### wikiann - croatian

In [19]:
data_name = "wikiann_hr"
wikiann_evaluation_output = model_evaluation.evaluate_model(wikiann_hr_words, wikiann_hr_labels)

  0%|          | 0/625 [00:00<?, ?it/s]

In [20]:
wikiann_seqeval = wikiann_evaluation_output.get_classification('Seqeval')
wikiann_seqeval

,Tag,Precision,Recall,F1,support
0,LOC,0.6860,0.7657,0.7237,4862
1,ORG,0.7359,0.4527,0.5606,4100
2,PER,0.7885,0.8590,0.8222,4404
3,micro,0.7344,0.7004,0.7170,13366
4,macro,0.7368,0.6925,0.7022,13366
5,weighted,0.7351,0.7004,0.7061,13366


In [21]:
wikiann_sklearn = wikiann_evaluation_output.get_classification('Sklearn')
wikiann_sklearn

,Tag,Precision,Recall,F1,support
0,B-LOC,0.7467,0.8178,0.7806,4862
1,B-ORG,0.8184,0.4893,0.6124,4100
2,B-PER,0.8517,0.9119,0.8808,4404
3,I-LOC,0.6557,0.4088,0.5036,2818
4,I-ORG,0.9180,0.3887,0.5462,7285
5,I-PER,0.9472,0.8130,0.8750,5675
6,O,0.8786,0.9855,0.9290,57070
7,accuracy,0.8680,86214,None,None
8,macro,0.8309,0.6879,0.7325,86214
9,weighted,0.8675,0.8680,0.8533,86214
